In [ ]:
import json
import pandas as pd
from collections import Counter
import math
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt
from ollama import chat


In [ ]:
data = [json.loads(line) for line in open("../results/predictions_union.jsonl").readlines()]

In [ ]:
df_data = [
    {
        "llm": sample["llm"],
        "setting": sample["setting"],
        "lang": sample["lang"],
        "doc_id": sample["doc_id"],
        "entity": pred["entity"],
        "roles": pred["predictions"],
        "target": pred["gold_labels"],
    }
    for sample in data
    for pred in sample["entities"]
    if sample["llm"] in [
        'gemma4:26b-a4b-it-q4_K_M', 
        'gemma4:e4b-it-q4_K_M', 
        'mistral-small3.2:24b-instruct-2506-q4_K_M', 
        'mistral:7b',
        'qwen3:30b-a3b-q4_K_M',
        'qwen3:4b-q4_K_M',
     ]
]

In [ ]:
df = pd.DataFrame(df_data)
ALL_ROLES = sorted(list({ri for r in list(df.roles) for ri in r}))

def compute_smoothed_p(values):
    dist = Counter(values)
    vec = np.zeros(len(ALL_ROLES))
    for role, v in dist.items():
        vec[ALL_ROLES.index(role)] = v

    # laplace smoothing
    alpha = 1e-15
    vec += alpha
    vec /= (vec.sum() + (alpha * vec.shape[0]))
    return vec
    

df["prob"] = df.roles.apply(compute_smoothed_p)
#df["freq_v"] = df.freq.apply(compute_roles_v)


#df["prob"] = df.freq.apply(lambda f: {k: v / sum(f.values()) for k, v in f.items()})
#df["prob_v"] = df["prob"].apply(compute_roles_v)

In [ ]:
df["entropy"] = df["prob"].apply(lambda p: -np.sum(p * np.log(p)))

In [ ]:
df_entropy = pd.DataFrame(df.groupby(["llm", "setting", "lang"]).entropy.mean()).reset_index().pivot(columns="lang", index=["llm", "setting"], values="entropy")
print(df_entropy.to_latex(float_format="%.3f"))

In [ ]:
dists = []
for group, rows in df.groupby(["llm", "doc_id", "lang", "entity"]):
    baseline_p = rows[rows.setting == "baseline"].prob.iloc[0]

    for v in ['all_triples', 'entity_triples', 'verbalized_triples']:
        try:
            v_p = rows[rows.setting == v].prob.iloc[0]

            #score = np.sum(baseline_p * np.log(baseline_p / v_p))
            score = np.sqrt(1 - np.sum(np.sqrt(baseline_p * v_p)))
            
            #sim = 1 + np.log(np.sum(np.sqrt(baseline_p * v_p)) + 1e-15)
            
            dists.append((*group, v, rows[rows.setting == v].roles.iloc[0], score))
        except:
            pass

dists_df = pd.DataFrame(dists, columns=["llm", "doc_id", "lang", "entity", "setting", "roles", "score"])

In [ ]:
hellinger_df = pd.DataFrame(dists_df.groupby(["llm", "setting"]).score.mean()).reset_index().pivot(columns="setting", index="llm", values="score")
print(hellinger_df.to_latex(float_format="%.3f"))